# 05 OpenOximetry Audit (Phase 1)

Implement schema and coverage audit with duplicate-header normalization for `bloodgas.csv`.

Inputs from: OpenOximetry source tables under `data/physionet.org/files/openox-repo/1.1.1`.
Outputs to: `outputs/openox_encounter_coverage.csv`, `outputs/openox_table_row_summary.csv`, and `outputs/openox_bloodgas_schema_notes.md`.
Next: notebook 06 for timing alignment and PPG encounter eligibility.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing data/physionet.org/files/openox-repo/1.1.1 and notebooks/")

OPENOX_DIR = (PROJECT_ROOT / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1").resolve()
OUTPUTS_DIR = (PROJECT_ROOT / "outputs").resolve()
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

ENCOUNTER_CSV = OPENOX_DIR / "encounter.csv"
PATIENT_CSV = OPENOX_DIR / "patient.csv"
PULSEOX_CSV = OPENOX_DIR / "pulseoximeter.csv"
SPECTRO_CSV = OPENOX_DIR / "spectrophotometer.csv"
BLOODGAS_CSV = OPENOX_DIR / "bloodgas.csv"

for path in [ENCOUNTER_CSV, PATIENT_CSV, PULSEOX_CSV, SPECTRO_CSV, BLOODGAS_CSV]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required OpenOximetry table: {path}")

OPENOX_ENCOUNTER_COVERAGE_CSV = OUTPUTS_DIR / "openox_encounter_coverage.csv"
OPENOX_TABLE_ROW_SUMMARY_CSV = OUTPUTS_DIR / "openox_table_row_summary.csv"
OPENOX_BLOODGAS_SCHEMA_NOTES_MD = OUTPUTS_DIR / "openox_bloodgas_schema_notes.md"

print(f"Project root: {PROJECT_ROOT}")
print(f"OpenOximetry dir: {OPENOX_DIR}")
print(f"Outputs dir: {OUTPUTS_DIR}")

## Load Core Tables

The bloodgas loader normalizes duplicate header names to avoid silent field loss.

In [ ]:
encounter_df = pd.read_csv(ENCOUNTER_CSV)
patient_df = pd.read_csv(PATIENT_CSV)
pulseox_df = pd.read_csv(PULSEOX_CSV)
spectro_df = pd.read_csv(SPECTRO_CSV)

with BLOODGAS_CSV.open("r", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    raw_headers = next(reader)

normalized_headers = []
header_counts = {}
for raw_name in raw_headers:
    base_name = str(raw_name).strip()
    if base_name == "":
        base_name = "unnamed"
    seen_count = header_counts.get(base_name, 0) + 1
    header_counts[base_name] = seen_count
    if seen_count == 1:
        normalized_headers.append(base_name)
    else:
        normalized_headers.append(f"{base_name}__dup{seen_count}")

bloodgas_df = pd.read_csv(
    BLOODGAS_CSV,
    skiprows=1,
    names=normalized_headers,
    low_memory=False,
    encoding="utf-8-sig",
)

duplicate_header_summary = [
    {"column_name": name, "count": count}
    for name, count in sorted(header_counts.items())
    if count > 1
]
duplicate_header_df = pd.DataFrame(duplicate_header_summary)

table_map = {
    "encounter": encounter_df,
    "patient": patient_df,
    "pulseoximeter": pulseox_df,
    "spectrophotometer": spectro_df,
    "bloodgas": bloodgas_df,
}

print({name: len(df) for name, df in table_map.items()})
print(f"Duplicate bloodgas headers found: {len(duplicate_header_df)}")

## Build Coverage Summaries And Write Artifacts

Aggregate table-level row coverage, build encounter-level completeness flags, and persist schema notes required by downstream OpenOximetry stages.

In [ ]:
table_row_summary_rows = []
for table_name, df in table_map.items():
    row = {
        "table_name": table_name,
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "unique_encounter_count": int(df["encounter_id"].nunique()) if "encounter_id" in df.columns else np.nan,
        "unique_patient_count": int(df["patient_id"].nunique()) if "patient_id" in df.columns else np.nan,
    }
    table_row_summary_rows.append(row)

table_row_summary_df = pd.DataFrame(table_row_summary_rows).sort_values("table_name").reset_index(drop=True)
table_row_summary_df.to_csv(OPENOX_TABLE_ROW_SUMMARY_CSV, index=False)

encounter_keys = encounter_df[["encounter_id", "patient_id"]].drop_duplicates().copy()
encounter_keys["has_patient_row"] = encounter_keys["patient_id"].isin(set(patient_df["patient_id"].dropna()))
encounter_keys["has_pulseox_rows"] = encounter_keys["encounter_id"].isin(set(pulseox_df["encounter_id"].dropna()))
encounter_keys["has_bloodgas_rows"] = encounter_keys["encounter_id"].isin(set(bloodgas_df["encounter_id"].dropna()))
encounter_keys["has_spectro_rows"] = encounter_keys["encounter_id"].isin(set(spectro_df["encounter_id"].dropna()))
encounter_keys["core_tables_present"] = (
    encounter_keys["has_patient_row"]
    & encounter_keys["has_pulseox_rows"]
    & encounter_keys["has_bloodgas_rows"]
    & encounter_keys["has_spectro_rows"]
)

encounter_keys = encounter_keys.sort_values("encounter_id").reset_index(drop=True)
encounter_keys.to_csv(OPENOX_ENCOUNTER_COVERAGE_CSV, index=False)

notes_lines = [
    "# OpenOximetry bloodgas.csv schema notes",
    "",
    f"Source: {BLOODGAS_CSV}",
    "",
    f"Raw header count: {len(raw_headers)}",
    f"Normalized header count: {len(normalized_headers)}",
    f"Duplicate header base names: {len(duplicate_header_df)}",
    "",
]

if duplicate_header_df.empty:
    notes_lines.extend([
        "No duplicate headers were found.",
        "",
    ])
else:
    notes_lines.extend([
        "## Duplicate header names and counts",
        "",
        "| column_name | count |",
        "|---|---:|",
    ])
    for row in duplicate_header_df.itertuples(index=False):
        notes_lines.append(f"| {row.column_name} | {int(row.count)} |")
    notes_lines.append("")

notes_lines.extend([
    "## First 40 normalized bloodgas headers",
    "",
])
for idx, col in enumerate(normalized_headers[:40], start=1):
    notes_lines.append(f"{idx}. {col}")

OPENOX_BLOODGAS_SCHEMA_NOTES_MD.write_text("\n".join(notes_lines) + "\n", encoding="utf-8")

print(f"Wrote: {OPENOX_TABLE_ROW_SUMMARY_CSV}")
print(f"Wrote: {OPENOX_ENCOUNTER_COVERAGE_CSV}")
print(f"Wrote: {OPENOX_BLOODGAS_SCHEMA_NOTES_MD}")

summary_snapshot = pd.DataFrame(
    [
        {"metric": "encounter_rows", "value": int(len(encounter_df))},
        {"metric": "encounter_unique_ids", "value": int(encounter_df["encounter_id"].nunique())},
        {"metric": "patient_rows", "value": int(len(patient_df))},
        {"metric": "pulseox_rows", "value": int(len(pulseox_df))},
        {"metric": "pulseox_unique_encounters", "value": int(pulseox_df["encounter_id"].nunique())},
        {"metric": "bloodgas_rows", "value": int(len(bloodgas_df))},
        {"metric": "bloodgas_unique_encounters", "value": int(bloodgas_df["encounter_id"].nunique())},
        {"metric": "spectro_rows", "value": int(len(spectro_df))},
        {"metric": "spectro_unique_encounters", "value": int(spectro_df["encounter_id"].nunique())},
        {"metric": "encounters_with_all_core_tables", "value": int(encounter_keys["core_tables_present"].sum())},
    ]
)
summary_snapshot